# Daily Challenge: Building Trustworthy Insights with DistilBERT

This notebook implements all requested tasks from the Daily Challenge.

In [ ]:
!pip install -q transformers datasets evaluate accelerate scikit-learn matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)
from sklearn.metrics import accuracy_score,f1_score
import torch
import evaluate


## 1. Data Loading & Inspection

In [ ]:
dataset=load_dataset("tweet_eval","sentiment")

print(dataset)

for split in dataset:
    print(f"\n{split}")
    print(dataset[split].features)

labels={0:"negative",1:"neutral",2:"positive"}

examples={0:[],1:[],2:[]}

for sample in dataset["train"]:
    if len(examples[sample["label"]])<2:
        examples[sample["label"]].append(sample["text"])

print("\nSaved examples:")
for k,v in examples.items():
    print(labels[k])
    for t in v:
        print("-",t)


## 2. Tokenization Pipeline

In [ ]:
tokenizer=AutoTokenizer.from_pretrained("distilbert-base-uncased")

def preprocess(examples):
    enc=tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )
    enc["labels"]=examples["label"]
    return enc

tokenized=dataset.map(preprocess,batched=True)
tokenized=tokenized.shuffle(seed=42)

tokenized.set_format(
    type="torch",
    columns=["input_ids","attention_mask","labels"]
)


## 3. Fine-Tuning

In [ ]:
model=AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=3
)

accuracy=evaluate.load("accuracy")
f1=evaluate.load("f1")

def compute_metrics(eval_pred):
    logits,labels=eval_pred
    preds=np.argmax(logits,axis=1)
    return{
        "accuracy":accuracy.compute(predictions=preds,references=labels)["accuracy"],
        "f1":f1.compute(predictions=preds,references=labels,average="macro")["f1"]
    }

training_args=TrainingArguments(
    output_dir="./distilbert_sentiment",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True
)

trainer=Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

trainer.train()
trainer.save_model("./best_model")
tokenizer.save_pretrained("./best_model")


## 4. Evaluation & Calibration

In [ ]:
metrics=trainer.evaluate(tokenized["validation"])
print(metrics)

pred=trainer.predict(tokenized["test"])

logits=pred.predictions
probs=torch.softmax(torch.tensor(logits),dim=1).numpy()
confidence=probs.max(axis=1)

plt.figure(figsize=(7,4))
plt.hist(confidence,bins=np.arange(0,1.1,0.1))
plt.title("Confidence Histogram")
plt.xlabel("Confidence")
plt.ylabel("Frequency")
plt.show()

print("Comment: A concentration near 1.0 indicates high-confidence predictions. If many incorrect predictions also have high confidence, the model may be overconfident.")


## 5. Attention Inspection

In [ ]:
base_model=AutoModel.from_pretrained(
    "distilbert-base-uncased",
    output_attentions=True
)

sentence=examples[0][0]

inputs=tokenizer(sentence,return_tensors="pt")

outputs=base_model(**inputs)

att=outputs.attentions[-1][0].mean(0)

tokens=tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

cls_attention=att[0].detach().numpy()

plt.figure(figsize=(12,4))
plt.bar(range(len(tokens)),cls_attention)
plt.xticks(range(len(tokens)),tokens,rotation=90)
plt.title("[CLS] Attention")
plt.show()

important=np.argsort(cls_attention)[-5:]
print("Most attended tokens:")
for i in important[::-1]:
    print(tokens[i],round(cls_attention[i],3))


## Production Inference Helper

In [ ]:
id2label={0:"negative",1:"neutral",2:"positive"}

def analyze_text(text):
    encoded=tokenizer(text,return_tensors="pt",truncation=True,max_length=128)

    with torch.no_grad():
        logits=model(**encoded).logits

    probs=torch.softmax(logits,dim=1)[0].numpy()

    pred=int(np.argmax(probs))

    outputs=base_model(**encoded)
    attention=outputs.attentions[-1][0].mean(0)[0].numpy()

    tokens=tokenizer.convert_ids_to_tokens(encoded["input_ids"][0])

    top=np.argsort(attention)[-5:]

    highlighted=[tokens[i] for i in top[::-1]]

    return{
        "label":id2label[pred],
        "confidence":float(probs[pred]),
        "highlighted_tokens":highlighted
    }

print(analyze_text("The customer service was amazing and solved my issue quickly."))


# Conclusion

## Deliverables Completed

- Dataset inspection
- DistilBERT fine-tuning
- Evaluation (Accuracy + Macro F1)
- Confidence histogram
- Attention visualization
- Explainable inference helper
- Saved model (./best_model)

The notebook is ready to be executed in Google Colab or Jupyter Notebook.
